# P06 — recover the interrupted development pilot

Use the SAME existing Colab GPU runtime. This notebook reads the interrupted run at `/content/P06_BFCL_Runs/20261029T115448Z-6d9a8339`, creates a new recovery run, copies only valid saved model records and generates missing/exception records. The original run, logs and failure ZIP are preserved. No model-data/cache deletion or dependency reinstall is performed.

Serving cap: 24576; other memory controls stay at the previous T4 profile. A new server profiles available KV cache at startup; exact fit remains to be confirmed. The handler enforces the same cap while preserving the official 4096-token answer allowance. No history truncation or automatic answer shortening is allowed.

Expected from the supplied console output: vanilla completed, generic control encountered one context exception, and schema-evidence had not started. The recovery uses actual saved files rather than assuming those counts. A record with an inference exception is not reusable. Task failures without infrastructure exceptions remain legitimate model outcomes and ARE reused.

Reused and newly generated outputs are recorded in `recovery_provenance.json`. This is mixed-attempt development evidence, not a single uniform-run leaderboard result or a fair latency experiment. The new cap and wrapper are versioned as infrastructure repairs; prompts, tools, reference trajectories and scorer remain unchanged. Upload the recovery output ZIP after completion (or export it if another logged failure occurs).


In [ ]:
import subprocess, sys, os, json, pathlib, datetime, platform
print('UTC', datetime.datetime.now(datetime.timezone.utc).isoformat())
print('Python', sys.version, 'Platform',platform.platform())
subprocess.run(['nvidia-smi'],check=True)



In [ ]:
source_run=pathlib.Path('/content/P06_BFCL_Runs/20261029T115448Z-6d9a8339')
assert source_run.is_dir(), 'Original run is missing. Keep the existing runtime, or provide its diagnostic ZIP before recovery.'
from pathlib import Path
import shutil, hashlib, uuid, signal
run_id=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
run_root=Path('/content/P06_BFCL_Runs')/run_id
run_root.mkdir(parents=True,exist_ok=False)
run_status={'setup':'pending','selection':'pending','generation':'pending','evaluation':'pending'}
model='Qwen/Qwen3-4B-Instruct-2507-FC'
policies=('vanilla','generic_control','schema_evidence')
runner_path=run_root/'p06_policy_runner.py'
runner_path.write_text('"""Development-only prompting policies; official scoring and tools are unchanged."""\nimport copy\nimport os\nimport runpy\n\nCOMMON = (\'Use the available tools and the conversation history to satisfy the user request. \'\n          \'A tool introduced by a later message is available from that message onward. \')\nPOLICIES = {\n    \'vanilla\': \'\',\n    \'generic_control\': COMMON + (\n        \'Work carefully through the task, review relevant details and check your progress. \'\n        \'Use concise explanations where helpful. Keep the requested outcome in mind \'\n        \'throughout the conversation and review whether the task is complete before reporting success.\'),\n    \'schema_evidence\': COMMON + (\n        \'Before acting, check the requested operation, the documented tool capabilities, \'\n        \'argument requirements and known conversation state. Do not substitute an operation \'\n        \'with different effects. Ask for clarification when information essential to the \'\n        \'requested outcome is unresolved; optional defaults alone do not establish intent. \'\n        \'If no available tool supports the operation, explain the limitation. \'\n        \'Claim completion only when supported by executed tool results.\')\n}\n\ndef make_formatter(original, policy):\n    instruction = POLICIES[policy]\n    def format_prompt(self, messages, functions):\n        if not instruction:\n            return original(self, messages, functions)\n        updated = copy.deepcopy(messages)\n        if updated and updated[0].get(\'role\') == \'system\':\n            updated[0][\'content\'] = instruction + \'\\n\\n\' + updated[0][\'content\']\n        else:\n            updated.insert(0, {\'role\': \'system\', \'content\': instruction})\n        return original(self, updated, functions)\n    return format_prompt\n\ndef make_context_guard(original):\n    def query(self, inference_data):\n        limit=int(os.environ[\'P06_CONTEXT_CAP\'])\n        self.max_context_length=min(self.max_context_length,limit)\n        prompt=self._format_prompt(inference_data[\'message\'],inference_data[\'function\'])\n        count=len(self.tokenizer.tokenize(prompt))\n        # Preserve the official 4096-token output allowance. No history truncation.\n        if count+4096+2>limit:\n            raise RuntimeError(f\'CONTEXT_BUDGET_EXHAUSTED: input={count}, answer_budget=4096, cap={limit}; no truncation or reduced budget applied\')\n        return original(self,inference_data)\n    return query\n\ndef main():\n    from bfcl_eval.model_handler.local_inference.qwen_fc import QwenFCHandler\n    from bfcl_eval.model_handler.local_inference.base_oss_handler import OSSHandler\n    OSSHandler._query_prompting=make_context_guard(OSSHandler._query_prompting)\n    policy = os.environ[\'P06_POLICY\']\n    QwenFCHandler._format_prompt = make_formatter(QwenFCHandler._format_prompt, policy)\n    runpy.run_module(\'bfcl_eval\', run_name=\'__main__\')\n\nif __name__ == \'__main__\':\n    main()\n')
config={}
server_max_len=24576
root=Path('/content/gorilla')
if not root.exists():
    subprocess.run(['git','clone','--filter=blob:none','https://github.com/ShishirPatil/gorilla.git',str(root)],check=True)
commit='6ea57973c7a6097fd7c5915698c54c17c5b1b6c8'
subprocess.run(['git','-C',str(root),'checkout',commit],check=True)
work=root/'berkeley-function-call-leaderboard'
print('Commit:',subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True).strip())
# BFCL at this commit pins numpy==1.26.4 and vllm==0.8.5.
# Use an isolated interpreter; the Colab notebook kernel may remain Python 3.13.
log_dir=run_root/'logs'
log_dir.mkdir(exist_ok=True)
print('This run:',run_root,flush=True)
def export_outputs():
    # Each run uses a new staging directory, so old result/score files cannot leak in.
    bundle=run_root/'export'
    bundle.mkdir(exist_ok=True)
    for sub in ('logs','result','score','conditions'):
        src=run_root/sub
        if src.exists():
            shutil.copytree(src,bundle/sub,dirs_exist_ok=True)
    for name in ('test_case_ids_to_generate.json','run_status.json','p06_policy_runner.py','recovery_provenance.json','source_attempt_outputs.zip'):
        src=run_root/name
        if src.exists(): shutil.copy2(src,bundle/name)
    manifest={'run_id':run_id,'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
              'bfcl_commit':commit,'model_id':model,'selected_ids':config,'conditions':list(policies),
              'status':run_status.copy(),'note':'Development-only 3-policy comparison on 2 families; not a final study','files':{}}
    for item in bundle.rglob('*'):
        if item.is_file() and item.name!='manifest.json':
            manifest['files'][str(item.relative_to(bundle))]=hashlib.sha256(item.read_bytes()).hexdigest()
    (bundle/'manifest.json').write_text(json.dumps(manifest,indent=2))
    archive=shutil.make_archive(str(run_root/'P06_BFCL_Development_Pilot_Recovery_Outputs'),'zip',bundle)
    print('Diagnostic/output archive:',archive,flush=True)
    return archive

def save_status():
    (run_root/'run_status.json').write_text(json.dumps(run_status,indent=2))

def run_logged(cmd,name,cwd=None,env=None,timeout=3600):
    import threading
    log_path=log_dir/(name+'.log')
    print('Running:',' '.join(map(str,cmd)),flush=True)
    proc=None
    timer=None
    timed_out=[]
    def stop_group():
        timed_out.append(True)
        if proc is not None and proc.poll() is None:
            try: os.killpg(proc.pid,signal.SIGKILL)
            except ProcessLookupError: pass
    try:
        with log_path.open('w') as out:
            proc=subprocess.Popen(list(map(str,cmd)),cwd=cwd,env=env,
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
            timer=threading.Timer(timeout,stop_group)
            timer.daemon=True
            timer.start()
            for line in proc.stdout:
                print(line,end='',flush=True)
                out.write(line)
                out.flush()
            rc=proc.wait()
        if timed_out: raise TimeoutError(f'{name} exceeded {timeout} seconds; see {log_path}')
        if rc: raise subprocess.CalledProcessError(rc,cmd)
    except BaseException as exc:
        if proc is not None and proc.poll() is None:
            try: os.killpg(proc.pid,signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait()
        run_status[name]='failed: '+type(exc).__name__
        save_status()
        print('Saved failure log:',log_path,flush=True)
        export_outputs()
        raise
    finally:
        if timer is not None: timer.cancel()
    return log_path

# Read host memory/disk without loading model weights into the notebook kernel.
def host_resources():
    mem={}
    for line in Path('/proc/meminfo').read_text().splitlines():
        key,value=line.split(':',1)
        mem[key]=int(value.strip().split()[0])*1024
    disk=shutil.disk_usage('/content')
    return {'ram_total_gib':mem.get('MemTotal',0)/2**30,
            'ram_available_gib':mem.get('MemAvailable',0)/2**30,
            'disk_free_gib':disk.free/2**30}
resource_snapshot=host_resources()
(log_dir/'host_resources_setup.json').write_text(json.dumps(resource_snapshot,indent=2))
print('Host resources:',resource_snapshot,flush=True)
run_logged(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv'], 'gpu_resources_setup')

venv=Path('/content/bfcl-py312')
assert (venv/'bin/python').exists(), 'Existing BFCL environment is missing.'
bfcl_python=venv/'bin/python'
bfcl_cli=venv/'bin/bfcl'
run_logged([bfcl_python,'-c','import sys; print(sys.version); assert sys.version_info[:2] == (3,12)'], 'verify_python')
run_logged([bfcl_python,'-m','pip','check'], 'pip_check')
run_logged([bfcl_python,'-c','import numpy, torch, vllm; print("numpy", numpy.__version__); print("torch", torch.__version__); print("vllm", vllm.__version__); print("CUDA", torch.cuda.is_available())'], 'verify_imports')
# Exercise the complete CLI import chain before attempting GPU inference.
preflight_env=os.environ.copy()
preflight_env['PATH']=str(venv/'bin')+os.pathsep+preflight_env.get('PATH','')
preflight_env['BFCL_PROJECT_ROOT']=str(run_root)
preflight_env['P06_CONTEXT_CAP']=str(server_max_len)
run_logged([bfcl_python,'-c','import soundfile; from qwen_agent.llm import get_chat_model; import bfcl_eval.__main__; print("BFCL/Qwen-Agent imports OK; soundfile", soundfile.__version__)'], 'verify_cli_imports', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'generate','--help'], 'verify_generate_cli', cwd=work, env=preflight_env)
run_logged([bfcl_cli,'evaluate','--help'], 'verify_evaluate_cli', cwd=work, env=preflight_env)
# Download tokenizer/config only, then exercise vLLM's actual tokenizer wrapper.
tokenizer_check = """
import json, torch, transformers
from transformers import AutoTokenizer, AutoConfig
from vllm.transformers_utils.tokenizer import get_cached_tokenizer
model_id='Qwen/Qwen3-4B-Instruct-2507'
assert transformers.__version__ == '4.51.3', transformers.__version__
tok=AutoTokenizer.from_pretrained(model_id)
assert hasattr(tok, 'all_special_tokens_extended')
cached=get_cached_tokenizer(tok)
assert cached.encode('BFCL smoke test') == tok.encode('BFCL smoke test')
cfg=AutoConfig.from_pretrained(model_id)
print(json.dumps({'transformers':transformers.__version__, 'tokenizer':type(tok).__name__,
 'model_revision':getattr(cfg,'_commit_hash',None), 'gpu':torch.cuda.get_device_name(0),
 'capability':torch.cuda.get_device_capability(0), 'tokenizer_check':'PASS'}))
"""
run_logged([bfcl_python,'-c',tokenizer_check], 'verify_tokenizer', cwd=work, env=preflight_env)
run_logged([bfcl_python,'-m','pip','freeze'], 'pip_freeze')


run_status['setup']='passed'
save_status()


In [ ]:
assert run_status['setup']=='passed', 'Run setup successfully first.'
config={c:[c+'_2',c+'_3'] for c in ('multi_turn_base','multi_turn_miss_func','multi_turn_miss_param')}
(run_root/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
# Verify the official loader actually resolves all six requested records and references.
selection_check="""
import json, os
from transformers import AutoTokenizer
from bfcl_eval._llm_response_generation import build_handler
from bfcl_eval.constants.eval_config import TEST_IDS_TO_GENERATE_PATH
from bfcl_eval.utils import load_test_entries_from_id_file, load_ground_truth_entry
categories, entries=load_test_entries_from_id_file(TEST_IDS_TO_GENERATE_PATH)
requested=json.loads(TEST_IDS_TO_GENERATE_PATH.read_text())
expected={i for ids in requested.values() for i in ids}
assert len(entries)==len(expected)==6
assert {e['id'] for e in entries}==expected
tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')
handler=build_handler('Qwen/Qwen3-4B-Instruct-2507-FC',0.001)
counts={}
import importlib.util
spec=importlib.util.spec_from_file_location('p06_policy_runner',str(TEST_IDS_TO_GENERATE_PATH.parent/'p06_policy_runner.py'))
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
for case in entries:
    prompt=handler._format_prompt(case['question'][0],case['function'])
    count=max(len(tokenizer.tokenize(module.make_formatter(type(handler)._format_prompt,policy)(handler,case['question'][0],case['function']))) for policy in module.POLICIES)
    counts[case['id']]=count
print('Initial prompt token counts:',counts)
assert max(counts.values())+4096<=int(os.environ['P06_CONTEXT_CAP']), 'Initial prompt plus BFCL generation budget exceeds server context cap.'
for category, ids in requested.items():
    refs={e['id']:e for e in load_ground_truth_entry(category)}
    for case_id in ids:
        case=next(e for e in entries if e['id']==case_id)
        assert case_id in refs
        assert len(case['question'])==len(refs[case_id]['ground_truth'])
print('PASS: six selected inputs and corresponding reference trajectories resolved.')
"""
run_logged([bfcl_python,'-c',selection_check],'verify_selection',cwd=work,env=preflight_env)
run_status['selection']='passed'
save_status()
print('Selected smoke-test IDs:',config)
print('Do not extrapolate results from two development families.')

# Reuse only original, unmodified, non-exception model records under identical prompts.
expected_runner_sha='6d14ded2d6d21473d2c32cca8ccd10eddb0bd9dc22511571414e158b89a3869f'
assert hashlib.sha256((source_run/'p06_policy_runner.py').read_bytes()).hexdigest()==expected_runner_sha, 'Original policy script differs; inspect before mixing outputs.'
expected={case_id for ids in config.values() for case_id in ids}
provenance={'source_run':str(source_run),'recovery_run':str(run_root),'old_context_cap':12288,
            'new_context_cap':server_max_len,'note':'Mixed-attempt development evidence; not a uniform-run timing comparison.','reused':[],'rejected_inference_errors':[]}
for policy in policies:
    old_root=source_run/'conditions'/policy/'result'/model.replace('/','_')
    for path in old_root.rglob('*_result.json'):
        rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        reusable=[]
        for row in rows:
            assert row['id'] in expected
            if row.get('traceback') or str(row.get('result','')).startswith('Error during inference:'):
                provenance['rejected_inference_errors'].append({'policy':policy,'case_id':row['id']})
                continue
            assert isinstance(row.get('result'),list), 'Unknown result structure; inspect source archive.'
            reusable.append(row)
            provenance['reused'].append({'policy':policy,'case_id':row['id'],'source_file':str(path),'source_sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        if reusable:
            dest=run_root/'conditions'/policy/'result'/model.replace('/','_')/path.relative_to(old_root)
            dest.parent.mkdir(parents=True,exist_ok=True)
            dest.write_text(''.join(json.dumps(row)+'\n' for row in reusable))
(run_root/'recovery_provenance.json').write_text(json.dumps(provenance,indent=2))
print('Reusing',len(provenance['reused']),'records; omitted',len(provenance['rejected_inference_errors']),'inference exceptions.')

source_archive=source_run/'P06_BFCL_Development_Pilot_Outputs.zip'
assert source_archive.is_file(), 'Original diagnostic ZIP missing; run its export cell first.'
shutil.copy2(source_archive,run_root/'source_attempt_outputs.zip')


In [ ]:
# Start a resource-limited server via BFCL's official --skip-server-setup path.
# This avoids modifying BFCL source, prompts, decoding or the evaluator.
import signal, socket, time, urllib.request
model='Qwen/Qwen3-4B-Instruct-2507-FC'
model_checkpoint='Qwen/Qwen3-4B-Instruct-2507'
assert run_status['selection']=='passed', 'Validate selection first.'
os.environ['BFCL_PROJECT_ROOT']=str(run_root)
run_env=os.environ.copy()
run_env['PATH']=str(venv/'bin')+os.pathsep+run_env.get('PATH','')
# Remove stale remote endpoint overrides from earlier notebook experiments.
for key in ('REMOTE_OPENAI_BASE_URL','REMOTE_OPENAI_TOKENIZER_PATH','REMOTE_OPENAI_API_KEY'):
    run_env.pop(key, None)
run_env['LOCAL_SERVER_ENDPOINT']='127.0.0.1'
# Select a free local port rather than reuse a possibly occupied server.
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0))
    port=sock.getsockname()[1]
run_env['LOCAL_SERVER_PORT']=str(port)
run_env['VLLM_USE_V1']='0'
run_env['PYTHONUNBUFFERED']='1'
run_env['P06_CONTEXT_CAP']=str(server_max_len)
capability=json.loads(subprocess.check_output(
    [str(bfcl_python),'-c','import json, torch; print(json.dumps(list(torch.cuda.get_device_capability(0))))'],
    text=True))
server_dtype='half' if capability[0] < 8 else 'bfloat16'
# server_max_len was also used by the initial-prompt preflight.
server_cmd=[str(venv/'bin/vllm'),'serve',model_checkpoint,
    '--host','127.0.0.1','--port',str(port),'--dtype',server_dtype,
    '--tensor-parallel-size','1','--gpu-memory-utilization','0.85',
    '--max-model-len',str(server_max_len),'--max-num-seqs','1',
    '--swap-space','0','--cpu-offload-gb','0',
    '--enable-chunked-prefill','--max-num-batched-tokens','2048',
    '--enforce-eager','--seed','0']
cmd=[str(bfcl_cli),'generate','--model',model,'--run-ids',
     '--backend','vllm','--skip-server-setup','--num-threads','1','--include-input-log']
profile={'server_command':server_cmd,'generation_command':cmd,
         'dtype':server_dtype,'context_cap':server_max_len,'gpu_capability':capability,
         'gpu_memory_utilization':0.85,'cpu_swap_gib':0,'cpu_offload_gib':0,
         'chunked_prefill':True,'max_num_batched_tokens':2048,
         'host_resources_before_server':host_resources(),
         'note':'Resource-limited smoke profile. Overlong requests must be recorded as runtime errors, never silently truncated or treated as model failures.'}
(log_dir/'runtime_profile.json').write_text(json.dumps(profile,indent=2))
server_log=log_dir/'vllm_server.log'
print('Server command:', ' '.join(server_cmd), flush=True)
print('Server log:',server_log, flush=True)
run_logged(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv'], 'gpu_resources_before_server')
def check_generation_results(condition_root,policy):
    expected={case_id for ids in config.values() for case_id in ids}
    records=[]
    result_root=condition_root/'result'/model.replace('/','_')
    for path in result_root.rglob('*_result.json'):
        records.extend(json.loads(line) for line in path.read_text().splitlines() if line.strip())
    observed=[item['id'] for item in records]
    errors=[item['id'] for item in records
            if item.get('traceback') or str(item.get('result','')).startswith('Error during inference:')]
    check={'expected_ids':sorted(expected),'observed_ids':observed,'inference_error_ids':errors}
    (log_dir/('result_integrity_'+policy+'.json')).write_text(json.dumps(check,indent=2))
    if set(observed)!=expected or len(observed)!=len(expected):
        raise RuntimeError('Missing, duplicate or unexpected result records; inspect result_integrity.json')
    if errors:
        raise RuntimeError('BFCL saved inference errors despite CLI success; inspect raw results and server log before scoring.')
server=None
run_status['generation']='running'
save_status()
try:
    with server_log.open('w') as out:
        server=subprocess.Popen(server_cmd,stdout=out,stderr=subprocess.STDOUT,
                                env=run_env,cwd=work,start_new_session=True)
        deadline=time.monotonic()+1800
        next_update=0
        while True:
            if server.poll() is not None:
                print(server_log.read_text(errors='replace')[-16000:])
                raise RuntimeError(f'vLLM exited with {server.returncode}; see {server_log}')
            try:
                with urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models',timeout=2) as response:
                    served=json.load(response)
                if model_checkpoint not in [item['id'] for item in served['data']]:
                    raise RuntimeError('Unexpected model served on selected port')
                print('Server ready.',flush=True)
                break
            except (OSError, ValueError):
                pass
            now=time.monotonic()
            if now>=deadline:
                print(server_log.read_text(errors='replace')[-16000:])
                raise TimeoutError(f'Server not ready after 30 minutes; see {server_log}')
            if now>=next_update:
                print('Waiting for model server; downloading/loading may take several minutes.',flush=True)
                next_update=now+30
            time.sleep(2)
        profile['policy_commands']={}
        for policy in policies:
            condition_root=run_root/'conditions'/policy
            condition_root.mkdir(parents=True,exist_ok=True)
            (condition_root/'test_case_ids_to_generate.json').write_text(json.dumps(config,indent=2))
            policy_env=run_env.copy()
            policy_env['BFCL_PROJECT_ROOT']=str(condition_root)
            policy_env['P06_POLICY']=policy
            policy_cmd=[str(bfcl_python),str(runner_path)]+cmd[1:]
            profile['policy_commands'][policy]=policy_cmd
            (log_dir/'runtime_profile.json').write_text(json.dumps(profile,indent=2))
            print('Condition:',policy,flush=True)
            run_logged(policy_cmd,'generate_'+policy,cwd=work,env=policy_env)
            check_generation_results(condition_root,policy)
            run_status['generation_'+policy]='passed'
            save_status()
except BaseException as exc:
    run_status['generation']='failed: '+type(exc).__name__
    save_status()
    export_outputs()
    raise
finally:
    # Stop only the process group created by this cell, including worker children.
    if server is not None:
        try:
            os.killpg(server.pid,signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(server.pid,signal.SIGKILL)
        except ProcessLookupError:
            pass
        server.wait()
        (log_dir/'host_resources_after_server.json').write_text(json.dumps(host_resources(),indent=2))

run_status['generation']='passed'
save_status()


In [ ]:
assert run_status['generation']=='passed', 'Resolve generation/inference errors before evaluating.'
categories=','.join(config)
for policy in policies:
    condition_root=run_root/'conditions'/policy
    eval_env=os.environ.copy()
    eval_env['BFCL_PROJECT_ROOT']=str(condition_root)
    eval_env['PATH']=str(venv/'bin')+os.pathsep+eval_env.get('PATH','')
    cmd=[str(bfcl_cli),'evaluate','--model',model,'--test-category',categories,'--partial-eval']
    try:
        run_logged(cmd,'evaluate_'+policy,cwd=work,env=eval_env)
        assert any((condition_root/'score').rglob('*.csv')), 'No score CSV produced.'
        run_status['evaluation_'+policy]='passed'
        save_status()
    except BaseException as exc:
        run_status['evaluation']='failed: '+type(exc).__name__
        save_status()
        export_outputs()
        raise
run_status['evaluation']='passed'
save_status()


In [ ]:
# This cell also works after a logged failure; the manifest records incomplete stages.
from google.colab import files
files.download(export_outputs())
